<a href="https://colab.research.google.com/github/malik09-teach/encoder_decoder_text_sumaraization_nlp/blob/main/text_summaraization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROJECT TEXT SUMMARAIZATION

## IMPORTANT LIBS

In [12]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import pandas as pd

print(f"TensorFlow Version: {tf.__version__}")
print(f"TFDS Version: {tfds.__version__}")

TensorFlow Version: 2.19.0
TFDS Version: 4.9.9


## LOADING DATA FROM KAGGEL

In [13]:
from google.colab import userdata
userdata.get('keggel')

'KGAT_ad53afa543f6dbd36f4bbda2a9f82f0d'

In [14]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!ls ~/.kaggle # This will list the files in the folder to prove it worked!

kaggle.json


In [15]:
#!/bin/bash
!kaggle datasets download gowrishankarp/newspaper-text-summarization-cnn-dailymail --unzip

Dataset URL: https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail
License(s): CC0-1.0
 92% 464M/503M [00:06<00:00, 114MB/s] 
100% 503M/503M [00:06<00:00, 81.5MB/s]


In [16]:
data_train=pd.read_csv("/content/cnn_dailymail/train.csv")
data_train.head()

,id,article,highlights
0,0001d1afc246a7964130f43ae940af6bc6c57f01,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,0002095e55fcbd3a2f366d9bf92a95433dc305ef,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...
2,00027e965c8264c35cc1bc55556db388da82b07f,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t..."
3,0002c17436637c4fe1837c935c04de47adb18e9a,(CNN) -- With a breezy sweep of his pen Presid...,Nina dos Santos says Europe must be ready to a...
4,0003ad6ef0c37534f80b55b4235108024b407f0b,Fleetwood are the only team still to have a 10...,Fleetwood top of League One after 2-0 win at S...


In [17]:
data_val=pd.read_csv("/content/cnn_dailymail/validation.csv")
data_val.head()



,id,article,highlights
0,61df4979ac5fcc2b71be46ed6fe5a46ce7f071c3,"Sally Forrest, an actress-dancer who graced th...","Sally Forrest, an actress-dancer who graced th..."
1,21c0bd69b7e7df285c3d1b1cf56d4da925980a68,A middle-school teacher in China has inked hun...,Works include pictures of Presidential Palace ...
2,56f340189cd128194b2e7cb8c26bb900e3a848b4,A man convicted of killing the father and sist...,"Iftekhar Murtaza, 29, was convicted a year ago..."
3,00a665151b89a53e5a08a389df8334f4106494c2,Avid rugby fan Prince Harry could barely watch...,Prince Harry in attendance for England's crunc...
4,9f6fbd3c497c4d28879bebebea220884f03eb41a,A Triple M Radio producer has been inundated w...,Nick Slater's colleagues uploaded a picture to...


## DATA PRE PROCESSING / DATA CLEANING

In [19]:
import pandas as pd
import re

# 1. Define the cleaning function using Regex
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower() # Convert to lowercase
    text = re.sub(r'<.*?>', '', text) # Remove any HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text) # Remove punctuation and special characters
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

# 2. Assuming you loaded your data into a Pandas DataFrame called 'train_df'
# Adjust the column names 'article' and 'highlights' if your CSV uses different names!
print("Cleaning training articles...")
data_train['cleaned_article'] = data_train['article'].apply(clean_text)

print("Cleaning training summaries and adding special tokens...")
# We add the SOS and EOS tokens directly to the summaries
data_train['cleaned_summary'] = data_train['highlights'].apply(lambda x: '<sos> ' + clean_text(x) + ' <eos>')

# Let's look at the cleaned results!
print(data_train[['cleaned_article', 'cleaned_summary']].head(2))

Cleaning training articles...
Cleaning training summaries and adding special tokens...
                                     cleaned_article  \
0  by associated press published 1411 est 25 octo...   
1  cnn ralph mata was an internal affairs lieuten...   

                                     cleaned_summary  
0  <sos> bishop john folda of north dakota is tak...  
1  <sos> criminal complaint cop used his role to ...  


In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np


In [21]:

# 1. Keep your sequence length limits
max_article_len = 400
max_summary_len = 50

# 2. Add the Vocabulary Cap to fix the OOM crash!
MAX_VOCAB_SIZE = 20000

# --- Process the Articles ---
print("Building capped article vocabulary...")
x_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
x_tokenizer.fit_on_texts(data_train['cleaned_article'])

x_train_seq = x_tokenizer.texts_to_sequences(data_train['cleaned_article'])
x_train_padded = pad_sequences(x_train_seq, maxlen=max_article_len, padding='post', truncating='post')

# --- Process the Summaries ---
print("Building capped summary vocabulary...")
y_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
y_tokenizer.fit_on_texts(data_train['cleaned_summary'])

y_train_seq = y_tokenizer.texts_to_sequences(data_train['cleaned_summary'])
y_train_padded = pad_sequences(y_train_seq, maxlen=max_summary_len, padding='post', truncating='post')

# --- Save Vocabulary Sizes ---
x_vocab_size = min(len(x_tokenizer.word_index) + 1, MAX_VOCAB_SIZE)
y_vocab_size = min(len(y_tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

print("\n--- Tokenization Complete ---")
print(f"New Capped Article Vocab: {x_vocab_size}")
print(f"New Capped Summary Vocab: {y_vocab_size}")
print(f"Final X_train shape: {x_train_padded.shape}")
print(f"Final Y_train shape: {y_train_padded.shape}")

Building capped article vocabulary...
Building capped summary vocabulary...

--- Tokenization Complete ---
New Capped Article Vocab: 20000
New Capped Summary Vocab: 20000
Final X_train shape: (287113, 400)
Final Y_train shape: (287113, 50)


In [22]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

# --- Hyperparameters ---
# You can tweak these later, but these are solid starting numbers
latent_dim = 300      # The number of neurons in the LSTM layers (the "memory" size)
embedding_dim = 100   # The size of the word vectors
max_article_len = 400 # Must match the padding size from yesterday

# ==========================================
# Phase 1: THE ENCODER
# ==========================================

# 1. The Input Layer: Tells the model to expect an array of 400 integers
encoder_inputs = Input(shape=(max_article_len,))

# 2. The Embedding Layer: Converts those integers into dense vectors of meaning
# x_vocab_size comes from your loaded tokenizers!
enc_emb = Embedding(x_vocab_size, embedding_dim, trainable=True)(encoder_inputs)

# 3. The LSTM Layer: Reads the embedded words chronologically
# return_sequences=True: Keeps the output of every single word (we need this for Attention later)
# return_state=True: Gives us the final memory states (Hidden & Cell)
encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

# 4. Connect the layers
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

# We group the final memory states together to pass to the Decoder
encoder_states = [state_h, state_c]

print("| Encoder built successfully!")
print(f"Encoder Output Shape: {encoder_outputs.shape}")

| Encoder built successfully!
Encoder Output Shape: (None, 400, 300)


In [23]:
# ==========================================
# Phase 2: THE DECODER
# ==========================================

# 1. The Input Layer: Tells the model to expect the padded summary arrays
# (We use max_summary_len, which we set to 50 earlier)
decoder_inputs = Input(shape=(max_summary_len,))

# 2. The Embedding Layer: Converts summary integers into dense vectors
dec_emb_layer = Embedding(y_vocab_size, embedding_dim, trainable=True)
dec_emb = dec_emb_layer(decoder_inputs)

# 3. The LSTM Layer:
# CRITICAL STEP: We pass the encoder_states as the initial_state!
# This is how the Decoder knows what the article was about.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, dec_state_h, dec_state_c = decoder_lstm(dec_emb, initial_state=encoder_states)

# 4. The Dense (Linear) Layer:
# This layer has a neuron for every single word in your summary vocabulary.
# The 'softmax' activation converts the outputs into probabilities (e.g., 90% chance the next word is "the").
decoder_dense = Dense(y_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# ==========================================
# Phase 3: BRINGING IT TOGETHER
# ==========================================

# We define the full model by specifying its inputs and its final output
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

print(" Decoder built and connected successfully!")
model.summary() # This will print out a table showing your entire architecture

 Decoder built and connected successfully!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 400)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 400, 100)  │  2,000,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 50, 100)   │  2,000,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 400,      │    481,200 │ embedding[0][0]   │
│                     │ 300), (None,      │            │                   │
│                     │ 300), (None,      │            │                   │
│                     │ 300)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 50, 300), │    481,200 │ embedding_1[0][0… │
│                     │ (None, 300),      │            │ lstm[0][1],       │
│                     │ (None, 300)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 20000) │  6,020,000 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,982,400 (41.89 MB)

 Trainable params: 10,982,400 (41.89 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

# ==========================================
# 1. Teacher Forcing Data Slicing
# ==========================================
print("Slicing data for Teacher Forcing...")

# Decoder Input: All rows, all columns EXCEPT the last one
decoder_input_data = y_train_padded[:, :-1]

# Decoder Target: All rows, all columns EXCEPT the first one
decoder_target_data = y_train_padded[:, 1:]

print(f"New Decoder Input Shape: {decoder_input_data.shape}")
print(f"New Decoder Target Shape: {decoder_target_data.shape}")

# ==========================================
# 2. Re-linking the Model (with length 49)
# ==========================================
# ENCODER
encoder_inputs = Input(shape=(max_article_len,))
enc_emb = Embedding(x_vocab_size, embedding_dim, trainable=True)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# DECODER
# Notice the max_summary_len - 1 ! It now expects 49 tokens.
decoder_inputs = Input(shape=(max_summary_len - 1,))
dec_emb_layer = Embedding(y_vocab_size, embedding_dim, trainable=True)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
# We pass the encoder's memory states to the decoder
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(y_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# BRINGING IT TOGETHER
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# ==========================================
# 3. Compiling
# ==========================================
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

print("\n Model sliced, linked, and compiled successfully!")

Slicing data for Teacher Forcing...
New Decoder Input Shape: (287113, 49)
New Decoder Target Shape: (287113, 49)

 Model sliced, linked, and compiled successfully!


In [26]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. The Early Stopping safety net
early_stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=2)

# 2. The Save Checkpoint (Saves directly to your mounted Google Drive!)
checkpoint_path = '/content/drive/MyDrive/Seq2Seq_Summarization/best_model.keras'
checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    verbose=1,
    save_best_only=True, # Only saves when the model actually improves
    mode='min'
)

# 3. Set Hyperparameters
BATCH_SIZE = 16
EPOCHS = 10

print(f" Starting {EPOCHS}-epoch training run with auto-saving...")

# 4. The actual training loop
history = model.fit(
    [x_train_padded, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,
    callbacks=[early_stop, checkpoint]    # Added the checkpoint here!
)

print("\n Training run complete!")

 Starting 10-epoch training run with auto-saving...
Epoch 1/10
14356/14356 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 5.4842
Epoch 1: val_loss improved from None to 4.60483, saving model to /content/drive/MyDrive/Seq2Seq_Summarization/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Seq2Seq_Summarization/best_model.keras
14356/14356 ━━━━━━━━━━━━━━━━━━━━ 830s 58ms/step - loss: 5.0713 - val_loss: 4.6048
Epoch 2/10
14355/14356 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 4.4517
Epoch 2: val_loss improved from 4.60483 to 4.32907, saving model to /content/drive/MyDrive/Seq2Seq_Summarization/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Seq2Seq_Summarization/best_model.keras
14356/14356 ━━━━━━━━━━━━━━━━━━━━ 836s 58ms/step - loss: 4.3998 - val_loss: 4.3291
Epoch 3/10
14355/14356 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 4.1630
Epoch 3: val_loss improved from 4.32907 to 4.21204, saving model to /content/drive/MyDrive/Seq2Seq_Summarization/best

In [27]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
import numpy as np

print("Splitting architecture for Inference...")

# 1. The Standalone Encoder
# Takes the article array and outputs the final memory states
encoder_model = Model(inputs=encoder_inputs, outputs=[state_h, state_c])

# 2. The Standalone Decoder
# Create new input layers to hold the memory states from the previous step
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Reuse the exact embedding layer we just trained
dec_emb2 = dec_emb_layer(decoder_inputs)

# Reuse the exact LSTM layer we just trained, but feed it the new state inputs
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]

# Reuse the exact Dense layer to predict the final word
decoder_outputs2 = decoder_dense(decoder_outputs2)

# Assemble the standalone Decoder
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

print("✅ Inference models built successfully!")

Splitting architecture for Inference...
✅ Inference models built successfully!


In [28]:
# Mappings to convert numbers back to English words
reverse_target_word_index = y_tokenizer.index_word
reverse_source_word_index = x_tokenizer.index_word
target_word_index = y_tokenizer.word_index

In [29]:
def decode_sequence(input_seq):
    # 1. Get the initial memory states from the Encoder
    states_value = encoder_model.predict(input_seq, verbose=0)

    # 2. Create an empty target sequence and put the <sos> token in it
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_word_index['sos']

    stop_condition = False
    decoded_sentence = ''

    # 3. Loop to generate words step-by-step
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        # Get the index of the most probable next word
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')

        # If the word isn't the end token, add it to our sentence
        if sampled_word != 'eos':
            decoded_sentence += ' ' + sampled_word

        # Stop if we hit <eos> or max length
        if (sampled_word == 'eos' or len(decoded_sentence.split()) >= (max_summary_len - 1)):
            stop_condition = True

        # Update the target sequence to the word we just predicted for the next loop
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update the memory states
        states_value = [h, c]

    return decoded_sentence.strip()

In [30]:
# Helper function to convert the integer arrays back to readable articles
def seq2text(input_seq):
    newString=''
    for i in input_seq:
        if(i!=0):
            newString = newString + reverse_source_word_index.get(i, '') + ' '
    return newString

def seq2summary(input_seq):
    newString=''
    for i in input_seq:
        if((i!=0 and i!=target_word_index['sos']) and i!=target_word_index['eos']):
            newString = newString + reverse_target_word_index.get(i, '') + ' '
    return newString

print("--- TESTING THE AI ON 3 ARTICLES ---\n")
for i in range(3):
    print(f"ARTICLE {i+1}:")
    print(seq2text(x_train_padded[i]))
    print("\nACTUAL HUMAN SUMMARY:")
    print(seq2summary(y_train_padded[i]))
    print("\nAI PREDICTED SUMMARY:")
    # We have to reshape the input array to (1, max_article_len) for the model
    print(decode_sequence(x_train_padded[i].reshape(1, max_article_len)))
    print("\n" + "="*50 + "\n")

--- TESTING THE AI ON 3 ARTICLES ---

ARTICLE 1:
by associated press published est 25 october 2013 updated est 25 october 2013 the bishop of the fargo catholic diocese in north dakota has exposed potentially hundreds of church members in fargo grand and to the hepatitis a virus in late september and early october the state health department has issued an advisory of exposure for anyone who attended five churches and took bishop john pictured of the fargo catholic diocese in north dakota has exposed potentially hundreds of church members in fargo grand and to the hepatitis a state program manager molly howell says the risk is low but officials feel its important to alert people to the possible exposure the diocese announced on monday that bishop john is taking time off after being diagnosed with hepatitis a the diocese says he contracted the infection through contaminated food while attending a conference for newly bishops in italy last month symptoms of hepatitis a include fever tiredn

In [31]:
import pickle

# Pointing to the exact folder where your model is saved
save_path = '/content/drive/MyDrive/Seq2Seq_Summarization'

print("Saving the updated tokenizers to Google Drive...")

# Save the Article tokenizer
with open(f'{save_path}/x_tokenizer.pkl', 'wb') as f:
    pickle.dump(x_tokenizer, f)

# Save the Summary tokenizer
with open(f'{save_path}/y_tokenizer.pkl', 'wb') as f:
    pickle.dump(y_tokenizer, f)

print("✅ Tokenizers successfully saved!")

Saving the updated tokenizers to Google Drive...
✅ Tokenizers successfully saved!
